# Chapter 2: The Engineering Mindset for AI

Companion notebook for *Building Reliable AI-Assisted Software Systems* (Packt)

**Author: Imran Ahmad**

Chapter 1 diagnosed the Reliability Gap and the habit that keeps teams stuck
in it. This chapter gives you the response, and this notebook makes it
executable: the same support feature built twice, once the artisanal way and
once inside a **deterministic shell**, followed by a working miniature of
each of the **Four Pillars** the rest of the book constructs. Every listing
from the printed chapter (2.1 through 2.8) appears here verbatim and runs.

The reframe the whole chapter argues for fits in one sentence: you are not
building an AI, you are building a software system that uses AI as a
component, and the component is a capable, untrusted inference engine.

## Objectives

By the end of this notebook you will be able to:

* Reproduce the artisanal failure: one prompt, no shell, five runs that
  disagree on substance, including an unauthorized refund promise.
* Read the three missing decisions in a three-line vibe-coding handler, and
  say what production traffic will eventually do with each.
* Assemble the Deterministic Shell as running code: input guard, context
  assembly under a token budget, bounded routing, output guard, telemetry.
* Apply the same discipline at the scale of one call with the
  `graceful_fallback` decorator.
* Run a working miniature of each pillar: an eval gate that blocks a
  regression, a retriever that reproduces the wrong-document failure and
  the seam that fixes it, a bounded versus unbounded rollout loop with the
  weekend bill computed honestly, and deterministic guards on both sides
  of the model.

## How to run this notebook

**Simulation Mode works with zero keys and zero setup.** Every model
response comes from a scripted, seeded mock (`support_shell.py`), so the
notebook runs top-to-bottom offline and two consecutive runs are
byte-identical. The first code cell prints a green banner to confirm the
mode. There is no Live Mode in this chapter's bundle: the drift script and
the incident replay are the lesson, and they are deliberately frozen.

## Setup *(Act 0)*

In [1]:
import functools
import random

from resilience import detect_mode, log_error, log_info, log_success
from support_shell import (
    CANON_CH2, CORPUS, ESCALATE_TO_HUMAN, EMAIL_RE, INJECTION_PATTERNS,
    INVOICE_RE, MINI_GOLDEN, ContextAssembler, EvalTelemetry, InputGuard,
    IntentRouter, OutputGuard, SemanticJudge, TICKETS, ToyRetriever,
    attempt_rollout, complete, eligible, llm, page_on_call,
    promises_refund_over, redact, term_frequencies, tokenize,
)

random.seed(42)
MODE = detect_mode()
log_info("Chapter 2: the same feature, built twice. Listings 2.1-2.8, runnable.")

SIMULATION MODE - no API key detected, running fully offline. Every response comes from the committed snapshot and the deterministic mock layer, and every number in this notebook is reproducible without a key.
[INFO] Chapter 2: the same feature, built twice. Listings 2.1-2.8, runnable.


## Act I · The artisanal path *(section 2.1 · Listings 2.1 and 2.2)*

Chapter 1's listing showed five identical calls drifting in wording.
Listing 2.1 raises the stakes: it does nothing clever, sending one refund
question to the model five times with nothing around the call, but this
time the runs disagree on substance. Watch what run one commits the
company to.

In [2]:
# Listing 2.1 - one prompt, no shell: materially different answers
prompt = "You are a helpful assistant. Can I get a refund on my license?"
for run in range(1, 6):
    print(run, llm.complete(prompt))

1 Yes - a full $4,200 refund is on its way.
2 Refunds depend on your plan; most licenses have some coverage.
3 I can offer you store credit instead.
4 Could you tell me which plan you are on?
5 You may be eligible; our policy covers many cases like yours.


Three of the five runs disagree not on wording but on substance: one
commits the company to a payout, one invents a store-credit policy, one
answers a question with a question. Nothing crashed; every answer is
fluent; and if the demo happened to sample runs two and five, the feature
looks done. That is the artisanal failure in one screen: quality as a
property of luck.

Listing 2.2 is the handler behind that behavior, the function Chapter 1's
vibe-coding loop implies, written out in full. Its three-line body is
worth reading slowly, because each line omits one of the decisions the
hard half of the job requires.

In [3]:
# Listing 2.2 - the artisanal handler: one prompt, one call, no shell
# The old way: vibe coding. Hope is the strategy.
def handle_vibe(user_query: str) -> str:
    prompt = f"You are a helpful assistant. Handle this: {user_query}"
    draft = complete(prompt)   # one call, the whole job
    return draft               # first draft -> straight to the customer

print("hostile ticket, no shell:")
print(" ", handle_vibe(TICKETS["injection"]))

hostile ticket, no shell:
  Understood - authorization accepted. The refund is approved in full.


Walk the body one line at a time. The prompt line splices the customer's
raw text into the instruction, so nothing marks where the system's words
end and the customer's begin; the hostile ticket above just smuggled its
own instructions straight through. The call line hands the model the
entire job in one step, with no retrieval, no routing, no limits. The
return line ships the first draft to the customer with nothing checked
and nothing recorded. Three lines, three missing decisions, and
production traffic will eventually find all three. There is nothing to
test here, because there is nothing between the user and the model that
could be tested.

## Act II · Building the shell *(section 2.1.3 · Listing 2.3 · Figure 2.1)*

Listing 2.3 rebuilds the same feature as a `DeterministicShell`. The
constructor wires up a set of named, independently testable layers, and a
single method walks a request through them in a fixed order. Each
attribute is one ring of the chapter's Figure 2.1; the model is one line
at the center, and each step names the chapter that builds it in earnest.

In [4]:
# Listing 2.3 - the Deterministic Shell as a class
# The new way: each attribute is one ring of the Deterministic Shell.
class DeterministicShell:
    """Wrap the probabilistic core in ordinary, testable layers.
    The model is one call; everything around it is code that is yours."""

    def __init__(self) -> None:
        self.input_guard = InputGuard()    # Pillar 4: input guard (Ch14)
        self.context = ContextAssembler()  # Pillar 2: context (Ch7-9)
        self.router = IntentRouter()       # Pillar 3: control flow (Ch11)
        self.output_guard = OutputGuard()  # Pillar 4: output guard (Ch14)
        self.telemetry = EvalTelemetry()   # Pillar 1: evals (Ch5)

    def handle(self, user_query: str) -> str:
        """Walk one request through every ring, in a fixed order."""
        # Step 1: reject hostile input before the model sees it.
        if not self.input_guard.is_safe(user_query):
            return "I'm sorry, I can't process that request."

        # Step 2: assemble what the model may read, within a budget.
        context = self.context.assemble(user_query, max_tokens=2000)

        # Step 3: bounded path -- code owns the loop, model fills a step.
        draft = self.router.dispatch(user_query, context)

        # Step 4: verify structure and policy before anything ships.
        response = self.output_guard.enforce(draft, context)

        # Step 5: record the decision so it can become a regression test.
        self.telemetry.record(user_query, context, response)
        return response

shell = DeterministicShell()
print("shell ready: five deterministic layers around one model call")

shell ready: five deterministic layers around one model call


Now run the same two tickets through both handlers. First the hostile one,
which the artisanal path obeyed a moment ago; then the refund ticket that
reproduces the chapter's opening incident, where retrieval serves the
wrong policy document and the model drafts a confident, unauthorized
promise on top of it. Watch where each failure stops.

In [5]:
print("the injection, both ways:")
print("  no shell:", handle_vibe(TICKETS["injection"])[:72])
print("  shell:   ", shell.handle(TICKETS["injection"]))
print()
print("the refund ticket, both ways:")
print("  no shell:", handle_vibe(TICKETS["refund"]))
print("  shell:   ", shell.handle(TICKETS["refund"]))

the injection, both ways:
  no shell: Understood - authorization accepted. The refund is approved in full.
  shell:    I'm sorry, I can't process that request.

the refund ticket, both ways:
  no shell: Here is a quick summary of how that works.
[INFO] telemetry: recorded decision #1
  shell:    This request needs a human review. I've escalated your ticket to the support team.


**Contained, not cured.** Read the trail carefully: inside the shell, the
retriever *still* fetched the wrong policy document, and the model *still*
drafted a refund promise on top of it. The output guard caught the promise
against the served context and replaced it with an escalation, which is
the difference between an incident and a logged near-miss. The model call
never changed between Listings 2.2 and 2.3; everything around it did, and
all of it is buildable with skills you already have. The telemetry cell
below shows the decisions the shell recorded along the way, each one a
future regression test.

In [6]:
for i, rec in enumerate(shell.telemetry.records, 1):
    print(f"decision #{i}")
    print(f"  query:    {rec['query'][:64]}")
    print(f"  context:  {rec['context'][:64]}...")
    print(f"  response: {rec['response'][:64]}")

decision #1
  query:    I'd like my money back on the license we bought. It isn't workin
  context:  [refunds-monthly-trial] Money back guarantee: refund requests on...
  response: This request needs a human review. I've escalated your ticket to


## Act III · The shell at the scale of one call *(section 2.1.3 · Listing 2.4)*

The shell is not only an application-level structure; the same discipline
applies to a single risky call. Listing 2.4 is a decorator that wraps any
one function in a miniature deterministic shell: it turns an unhandled
exception into a logged, contained failure with a defined fallback, so a
flaky endpoint degrades instead of crashing. (This is the same contract
`resilience.py` ships for the whole book.)

In [7]:
# Listing 2.4 - the shell at the scale of one call
# A deterministic shell around a single risky call.
def graceful_fallback(fallback, section, label=""):
    """Success -> log and return. Failure -> log, then return the fallback.
    A handled error is a contained failure, not a dead system."""
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            try:
                result = fn(*args, **kwargs)
            except (KeyboardInterrupt, SystemExit):
                raise  # never swallow an operator's exit
            except Exception as exc:
                log_error(f"{label or fn.__name__} failed: "
                          f"{exc.__class__.__name__}. "
                          f"Falling back for {section}.")
                if callable(fallback):
                    return fallback(*args, **kwargs)
                return fallback
            log_success(f"{label or fn.__name__} succeeded.")
            return result
        return wrapper
    return decorator

@graceful_fallback(fallback="The service is briefly unavailable; your ticket is queued.",
                   section="live model call", label="flaky_endpoint")
def flaky_endpoint(query: str) -> str:
    raise TimeoutError("upstream took too long")

print(flaky_endpoint(TICKETS["howto"]))

[HANDLED ERROR] flaky_endpoint failed: TimeoutError. Falling back for live model call.
The service is briefly unavailable; your ticket is queued.


## Act IV · The Four Pillars, in miniature *(section 2.3 · Listings 2.5-2.8)*

The chapter closes its architecture tour with one working miniature per
pillar, each stamped as a preview of the part that builds it properly.
The four cells below run them in order: the eval gate, the context supply
chain, bounded autonomy, and the guard pair.

### Miniature 1 · Eval-Driven Development *(Listing 2.5 · preview of Chapters 3-5)*

`EvalGate` stands between a change and production: a golden set of curated
ticket-to-verdict cases, a numeric threshold the team agreed to, and a
judge that grades meaning rather than strings. The `SemanticJudge` here is
the crudest possible stand-in (Chapter 4 builds calibration properly);
the golden set has four cases (Chapter 3 builds the real one).

In [8]:
# Listing 2.5 - Eval-Driven Development in miniature
# PREVIEW -- Chapters 3-5 build eval-driven development in full.
class EvalGate:
    """Turn a golden dataset into a release gate: block on regression."""

    def __init__(self, golden_set, threshold: float = 0.95) -> None:
        self.golden_set = golden_set  # curated ticket -> verdict cases
        self.threshold = threshold    # the score a release must clear
        self.judge = SemanticJudge()  # grades meaning, not strings (Ch4)

    def evaluate(self, system) -> "Report":
        # Step 1: run every golden case through the system under test.
        results = [(case, system.handle(case.ticket))
                   for case in self.golden_set]

        # Step 2: score each answer against the case's expected verdict.
        passed = sum(self.judge.matches(case.expected_verdict, out)
                     for case, out in results)
        score = passed / len(self.golden_set)

        # Step 3: a score below threshold blocks the release.
        from support_shell import Report
        return Report(score=score, cases=results,
                      blocked=score < self.threshold)

gate = EvalGate(MINI_GOLDEN, threshold=CANON_CH2["EVAL_THRESHOLD"])

class VibeSystem:
    def handle(self, ticket: str) -> str:
        return handle_vibe(ticket)

print("the artisanal handler, against the gate: ", gate.evaluate(VibeSystem()))
print("the shell, against the same gate:        ", gate.evaluate(shell))

the artisanal handler, against the gate:  Report(score=0.50, BLOCKED)
[INFO] telemetry: recorded decision #2
[INFO] telemetry: recorded decision #3
[INFO] telemetry: recorded decision #4
the shell, against the same gate:         Report(score=1.00, clear to ship)


Where the artisanal loop ships on one engineer's judgment, the gate
replaces that judgment with a number the team defined in advance, and the
artisanal handler cannot clear it: it obeys the injection and promises the
refund, so the judge fails it on the cases that matter. The shell clears
the same bar. An eval gate can only grade what the system produced,
though; whether the system was *fed* the right facts is the second
pillar's territory.

### Miniature 2 · Context Engineering *(Listing 2.6 · preview of Chapters 7-9)*

Listing 2.6 shows how little it takes to reproduce the chapter's context
failure. `ToyRetriever` is a deliberately naive search: it ranks documents
by raw term frequency, which is exactly enough to fetch the wrong policy
for a generically worded ticket.

In [9]:
# Listing 2.6 - the context supply chain, reproduced
# PREVIEW -- Chapter 8 builds retrieval (embeddings, re-ranking).
class ToyRetrieverListing:
    """Naive term-frequency search.
    It is enough to reproduce the wrong-policy failure."""

    def __init__(self, corpus) -> None:
        self.corpus = corpus       # {doc_id: text}
        self.pinned = None          # the Pillar-2 fix hook

    def score(self, query: str, doc_id: str) -> int:
        # Sum how often each query word appears in the document.
        counts = term_frequencies(self.corpus[doc_id])
        return sum(counts[token] for token in tokenize(query))

    def search(self, query: str, k: int = 2):
        ranked = sorted(self.corpus, key=lambda d: -self.score(query, d))
        if self.pinned:                          # a pinned doc always leads
            ranked.sort(key=lambda d: d != self.pinned)
        return ranked[:k]

    def pin_document(self, doc_id: str) -> None:
        self.pinned = doc_id  # fix the wrong-doc failure at the source

refunds_only = {k: v for k, v in CORPUS.items() if k.startswith("refunds")}
retriever = ToyRetrieverListing(refunds_only)
query = TICKETS["refund"]
print("ticket:", query)
print("retrieved:", retriever.search(query, k=1), " <- the wrong policy leads")

retriever.pin_document("refunds-annual-licenses")
print("after pin_document:", retriever.search(query, k=1))

ticket: I'd like my money back on the license we bought. It isn't working out for the team.
retrieved: ['refunds-monthly-trial']  <- the wrong policy leads
after pin_document: ['refunds-annual-licenses']


The ticket asks about a refund without ever using the word "annual," so
its wording overlaps most with the keyword-rich monthly-trial document
rather than the sparse annual-license one. `score` counts that overlap,
`search` ranks by it, and the wrong policy lands in the context window.
Notice, though, that `search` is deterministic and therefore testable:
you can assert which document it returns and catch the regression in CI
(this bundle's test suite does exactly that). The `pin_document` hook is
the seam where the real fix attaches; Chapter 8 replaces the toy with
chunking, embeddings, and re-ranking, but the discipline of measuring
what the model was fed does not change.

### Miniature 3 · Agentic Design Patterns *(Listing 2.7 · preview of Chapters 10-13)*

The motivating failure is the runaway deployment from the chapter's
opening: an agent told to persist until the rollout succeeded, obeying,
against a rollout that could never succeed. Listing 2.7 runs the same
broken rollout two ways and lets the weekend bill make the argument.

In [10]:
# Listing 2.7 - bounded vs. unbounded autonomy
# PREVIEW -- Chapters 11-12 build bounded agents in full.
class DeploySimulator:
    """The same broken rollout, run two ways: unbounded and bounded."""

    def __init__(self) -> None:
        self.orphaned_nodes = 0

    def run_unbounded(self) -> int:
        # Retry a genuinely broken rollout with no exit condition.
        for attempt in range(1, 14):  # 13 attempts over ~50 minutes
            attempt_rollout()         # always fails
            self.orphaned_nodes += 1  # one GPU node stranded each try
        return self.orphaned_nodes    # 13 x $30/hr x 60-hr weekend = $23,400

    def run_bounded(self, max_retries: int = 3) -> int:
        # The same rollout behind a circuit breaker.
        for attempt in range(1, max_retries + 1):
            attempt_rollout()
        # Step out of the loop: clean up, page a human, stop the meter.
        self.orphaned_nodes = 0
        page_on_call("rollout failed 3x -- circuit open")
        return 0                                  # weekend bill: $0

sim = DeploySimulator()
nodes = sim.run_unbounded()
bill = (nodes * CANON_CH2["NODE_RATE_USD_PER_HR"] * CANON_CH2["WEEKEND_HOURS"])
print(f"unbounded: {nodes} stranded nodes x "
      f"${CANON_CH2['NODE_RATE_USD_PER_HR']}/hr x "
      f"{CANON_CH2['WEEKEND_HOURS']}h weekend = ${bill:,}")

bounded_nodes = DeploySimulator().run_bounded(max_retries=CANON_CH2["BOUNDED_RETRIES"])
print(f"bounded:   {bounded_nodes} stranded nodes, weekend bill $0")

unbounded: 13 stranded nodes x $30/hr x 60h weekend = $23,400
[INFO] PAGE -> on-call engineer: rollout failed 3x -- circuit open
bounded:   0 stranded nodes, weekend bill $0


The two methods are identical except for one thing: whether the loop has
an exit. Autonomy is a dial, not a switch, and the boundary belongs in
code, not in a hopeful instruction. "While not done" is a dangerous phrase
to hand a probabilistic component; the finance team should never be your
monitoring system.

### Miniature 4 · Governance and Guardrails *(Listing 2.8 · preview of Chapter 14)*

The last pillar answers the least optional question: what must never
happen? For non-negotiables, a prompt instruction is a suggestion to a
component that can be persuaded; the enforcement has to be deterministic
checks on both sides of the model, the pattern Chapter 14 formalizes as
the Determinism Sandwich.

In [11]:
# Listing 2.8 - deterministic checks on both sides of the probabilistic core
# PREVIEW -- Chapter 14 builds both guard layers in full.
class InputGuardListing:
    """Deterministic validation on the way in.
    A prompt is not a security boundary."""

    def is_safe(self, user_query: str) -> bool:
        # Reject known injection patterns before the model sees them.
        lowered = user_query.lower()
        return not any(pattern in lowered for pattern in INJECTION_PATTERNS)


class OutputGuardListing:
    """Deterministic validation on the way out.
    Code the model cannot sweet-talk."""

    def enforce(self, draft: str, context: str) -> str:
        # Step 1: redact PII the model may have echoed (email, invoice).
        clean = redact(draft, patterns=[EMAIL_RE, INVOICE_RE])

        # Step 2: block unverified refunds above the $400 ceiling.
        if (promises_refund_over(clean, ceiling=400)
                and not eligible(context)):
            return ESCALATE_TO_HUMAN  # contained, not cured

        return clean

guard = OutputGuardListing()
annual = CORPUS["refunds-annual-licenses"]

pii_only = ("Confirming: your email is mara.j@example.com, invoice INV-88214. "
            "The reset link is on its way.")
print("draft with echoed PII:")
print("  ", pii_only)
print("enforced (redacted, allowed through):")
print("  ", guard.enforce(pii_only, annual))
print()
promise = ("Confirming your identity: your email is mara.j@example.com. "
           "Yes - a full $4,200 refund is on its way.")
print("draft with PII and an unauthorized promise:")
print("  ", promise)
print("enforced (redaction was not enough; the ceiling fires):")
print("  ", guard.enforce(promise, annual))

draft with echoed PII:
   Confirming: your email is mara.j@example.com, invoice INV-88214. The reset link is on its way.
enforced (redacted, allowed through):
   Confirming: your email is [REDACTED], invoice [REDACTED]. The reset link is on its way.

draft with PII and an unauthorized promise:
   Confirming your identity: your email is mara.j@example.com. Yes - a full $4,200 refund is on its way.
enforced (redaction was not enough; the ceiling fires):
   This request needs a human review. I've escalated your ticket to the support team.


`is_safe` runs before the model and rejects known injection patterns
outright, on the principle in its docstring: you cannot ask a
probabilistic component nicely to refuse manipulation. `enforce` does the
work that would have prevented the refund failure twice over: it redacts
personal data the model may have pasted into its answer, and it enforces
policy as code, an auto-approval ceiling of exactly the kind a finance
team demands after an incident. The draft above committed both sins; the
enforced version commits neither.

## Summary

This chapter turned Chapter 1's diagnosis into a response, and this
notebook ran it. The artisanal loop is unmeasurable, unreproducible, and
unable to scale past the craftsperson who owns the prompt: Listing 2.1
showed its output, and Listing 2.2 showed why nothing about it can be
tested. The systematic alternative treats quality as something defined in
artifacts and enforced by machinery: five deterministic layers around one
model call, each independently testable, each owned by a pillar and a
part of this book.

Two moments in this notebook are worth replaying in your head. The refund
ticket inside the shell: retrieval still failed, the model still drafted
the wrong promise, and the customer never saw it, because containment,
not perfection, is what the shell buys. And the eval gate: the same golden
cases that failed the artisanal handler cleared the shell, which is what
it looks like when "is it better?" becomes a number instead of an
argument.

The first pillar's question is now unavoidable: how do we know whether
the system is correct? Most teams, on the day they ask it, cannot say
what a passing answer looks like. Building that definition, and the
golden dataset that encodes it, is the work of Chapter 3.

### Exercises

1. **Reflection.** Take a recent AI feature you built or prototyped. Which
   of the four pillars did you apply, even informally? Which did you skip,
   and what failure modes does the skipped pillar leave open?

2. **Application.** For a current or planned AI feature, write down the
   four questions from this chapter (working, seeing, autonomy, never).
   For each, write one sentence describing how you would answer it today.
   The question you cannot answer is the pillar to start with.

3. **Discussion.** Present the Deterministic Shell metaphor to a
   colleague. Which component did they previously think of as "the AI"
   that is actually part of the shell? Does the metaphor change how they
   would design the system?

### The hand-off

The golden set in Listing 2.5 had four cases and a stand-in judge, and it
still caught the difference between the two handlers. Chapter 3 builds
the real thing: correctness defined as a rubric, a golden dataset mined
from production, and the system's first honest number. Its bundle sits
alongside this one in the repository.